# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kenzo4k/Flyrank-ML/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window\n\n*One row = one what, over which dates? State it, then verify it below.*\n\n### Plain-Words Contract Definition (Lane 3: Structured Content Archetype Clustering):\n\n1. **Unit of Analysis (Grain)**: **1 row = 1 pseudonymized content item (`content_id` within `client_id`)**.\n2. **Primary Dataset & Upstream Warehouse**:\n   - **Primary Model Dataset**: `data/raw/content_refresh_anonymized.csv` (30,000 content items across 32 clients, 90-day aggregation window).\n   - **Upstream Data Warehouse**: `hf://datasets/FlyRank/internship-warehouse` (`dim_clients`, `dim_content`, `fact_content_daily_performance`).\n3. **Time Window**:\n   - **Observation Window**: Trailing 90-day search and analytics activity window ($T_0 = \\text{Day 90}$).\n   - **Evaluative Benchmark Window**: Trailing 30 days vs. preceding 30 days (`trend_pct` / `trend_direction`).\n4. **Target / Evaluative Proxy**:\n   - **Model Training**: **None** (Unsupervised Machine Learning — Clustering).\n   - **Evaluative Benchmark Proxy**: `is_declining_label` (binary decay indicator: $\\text{trend\\_direction} == \\text{'down'}$), reserved strictly as a downstream evaluation benchmark to verify whether discovered archetypes concentrate performance decay.\n5. **Deliberately Excluded**:\n   - **Future / Outcome Metrics**: `trend_direction`, `trend_pct`, `impressions_last_30d`, `impressions_prev_30d` (target leakage for decay evaluation).\n   - **Product Rule Flags**: `health_score`, `ctr_tier`, `impression_tier`, `position_tier`, `age_tier` (hand-crafted heuristic rules).\n   - **Entity Identifiers**: `client_id`, `content_id` (context only for `GroupShuffleSplit` cross-validation).\n

In [1]:
# Section 1 Code: Setup, Warehouse Authentication, and Grain/Window Verification
import os, sys, getpass
import duckdb
import pandas as pd
import numpy as np
from dotenv import load_dotenv

# Robust token resolution: env var -> local .env -> prompt
load_dotenv('.env')
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    for env_path in ['../.env', '../../.env']:
        if os.path.exists(env_path):
            load_dotenv(env_path)
            HF_TOKEN = os.environ.get('HF_TOKEN')
            if HF_TOKEN:
                break

con = duckdb.connect()
if HF_TOKEN:
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
CLIENTS_SRC = f"read_parquet('{REL}/dim_clients.parquet')"
CONTENT_SRC = f"read_parquet('{REL}/dim_content.parquet')"

print("=== WAREHOUSE DUCKDB CONNECTION ESTABLISHED ===")
client_count = con.sql(f"SELECT COUNT(*) FROM {CLIENTS_SRC}").fetchone()[0]
print(f" - dim_clients Verified : {client_count} portfolio clients connected")


=== WAREHOUSE DUCKDB CONNECTION ESTABLISHED ===


 - dim_clients Verified : 104 portfolio clients connected


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Field Classification (Knowable at the Decision Moment)

We establish the 7 core feature frame for Lane 3 (Structured Content Archetype Clustering):

1. **Feature Columns (7 Continuous & Structural Signals)**:
   - `impressions_90d`: Search volume demand (log1p transformed to handle portfolio skewness).
   - `avg_position`: Average keyword rank (with 0 imputed to 100 representing no Page 1-10 rankings).
   - `ctr`: Organic search click-through rate percentage ($0.76 = 0.76\%$).
   - `days_since_last_update`: Days elapsed since last editorial update (freshness signal).
   - `content_age_days`: Total days since initial publication (lifecycle maturity signal).
   - `word_count`: Article length in words (with median imputation and explicit `has_word_count` indicator).
   - `engagement_rate`: GA4 user engagement rate percentage.

   **Raw Feature to Model-Ready Feature Mapping**:
   | Ingested Raw Field | Preprocessing / Cleaning Step | Model Matrix Column Key |
   |---|---|---|
   | `impressions_90d` | $\log(1 + x)$ variance stabilization | `log_impressions` |
   | `avg_position` | Zero-imputation to unranked baseline ($0.0 \rightarrow 100.0$) | `avg_position` (cleaned inline) |
   | `ctr` | Preserved continuous percentage | `ctr` |
   | `days_since_last_update` | Preserved continuous integer days | `days_since_last_update` |
   | `content_age_days` | Preserved continuous integer days | `content_age_days` |
   | `word_count` | Median imputation + `has_word_count` flag | `word_count` (imputed inline) |
   | `engagement_rate` | Preserved continuous percentage | `engagement_rate` |

   *(Note: Model matrices in W04/W05 apply these transformations directly inline within the feature frame while retaining standard column identifiers `avg_position` and `word_count`).*

2. **Label / Evaluative Proxy Columns**:
   - `is_declining_label`: Evaluative benchmark derived from `trend_direction == 'down'`, strictly withheld from clustering.

3. **Context Columns**:
   - `content_id`, `client_id`: Entity keys used strictly for joins and grouped cross-validation splits.

4. **Excluded Columns (with Rationale)**:
   - `trend_direction`, `trend_pct`: Direct label leakage for decay risk evaluation.
   - `health_score`, `impression_tier`, `position_tier`, `freshness_tier`: Hand-crafted rules and discretizations that discard continuous variance.
   - `content_id`, `client_id`: Identifiers causing entity memorization.


In [2]:
# Section 2 Code: Field Bucket Verification & 7-Feature Frame Creation
feature_cols = [
    'impressions_90d', 'avg_position', 'ctr', 'days_since_last_update',
    'content_age_days', 'word_count', 'engagement_rate'
]
label_cols = ['is_declining_label']
context_cols = ['client_id', 'content_id']
excluded_cols = [
    'trend_direction', 'trend_pct', 'health_score',
    'impression_tier', 'position_tier', 'age_tier', 'freshness_tier',
    'client_id', 'content_id'
]

feat_leak_overlap = set(feature_cols).intersection(set(excluded_cols))
feat_label_overlap = set(feature_cols).intersection(set(label_cols))

print("=== DATA CONTRACT FIELD BUCKETS ===")
print(f" - Feature Columns (<= 7) : {len(feature_cols)} -> {feature_cols}")
print(f" - Evaluative Label Proxy : {len(label_cols)} -> {label_cols}")
print(f" - Context / Join Keys    : {len(context_cols)} -> {context_cols}")
print(f" - Excluded Columns       : {len(excluded_cols)} -> {excluded_cols}")

print("\n=== LEAKAGE & OVERLAP INTEGRITY CHECKS ===")
print(f" - Overlap (Features x Excluded): {list(feat_leak_overlap)} -> {'PASSED (Zero Leakage)' if len(feat_leak_overlap) == 0 else 'FAILED'}")
print(f" - Overlap (Features x Labels)  : {list(feat_label_overlap)} -> {'PASSED (Zero Leakage)' if len(feat_label_overlap) == 0 else 'FAILED'}")

# Load primary model dataset
data_candidates = [
    'data/raw/content_refresh_anonymized.csv',
    '../data/raw/content_refresh_anonymized.csv',
    '../../data/raw/content_refresh_anonymized.csv'
]
csv_path = next((p for p in data_candidates if os.path.exists(p)), None)
if csv_path is None:
    raise FileNotFoundError("Could not locate data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(csv_path)

df['is_declining_label'] = (df['trend_direction'].str.lower() == 'down').astype(int)
df['log_impressions'] = np.log1p(df['impressions_90d'])
df['clean_avg_position'] = df['avg_position'].replace(0, 100.0)
df['has_word_count'] = df['word_count'].notnull().astype(int)
df['clean_word_count'] = df['word_count'].fillna(df['word_count'].median())

feat_df = df[['log_impressions', 'clean_avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'clean_word_count', 'engagement_rate']]

print("\n=== SEVEN-FEATURE MATRIX SUMMARY ===")
print(f" - Feature Matrix Shape : {feat_df.shape[0]:,} rows x {feat_df.shape[1]} features (Lane 3 Feature Frame)")
print(f" - Evaluative Target Base Rate (is_declining_label): {df['is_declining_label'].mean()*100:.1f}%")
print("\nDescriptive Statistics:")
print(feat_df.describe().round(2).to_string())


=== DATA CONTRACT FIELD BUCKETS ===
 - Feature Columns (<= 7) : 7 -> ['impressions_90d', 'avg_position', 'ctr', 'days_since_last_update', 'content_age_days', 'word_count', 'engagement_rate']
 - Evaluative Label Proxy : 1 -> ['is_declining_label']
 - Context / Join Keys    : 2 -> ['client_id', 'content_id']
 - Excluded Columns       : 9 -> ['trend_direction', 'trend_pct', 'health_score', 'impression_tier', 'position_tier', 'age_tier', 'freshness_tier', 'client_id', 'content_id']

=== LEAKAGE & OVERLAP INTEGRITY CHECKS ===
 - Overlap (Features x Excluded): [] -> PASSED (Zero Leakage)
 - Overlap (Features x Labels)  : [] -> PASSED (Zero Leakage)



=== SEVEN-FEATURE MATRIX SUMMARY ===
 - Feature Matrix Shape : 30,000 rows x 7 features (Lane 3 Feature Frame)
 - Evaluative Target Base Rate (is_declining_label): 54.2%

Descriptive Statistics:
       log_impressions  clean_avg_position       ctr  days_since_last_update  content_age_days  clean_word_count  engagement_rate
count         30000.00            30000.00  30000.00                30000.00          30000.00          30000.00         30000.00
mean              6.19               20.36      0.51                   46.10            256.17           3048.54             2.53
std               2.69               22.04      3.28                   42.08            132.71           1256.27             8.31
min               0.69                0.10      0.00                    1.00             90.00              8.00             0.00
25%               4.41                6.80      0.00                   20.00            132.00           2621.00             0.00
50%               6.60  

## 3. Verify it with queries (grain, counts, missing values, windows)\n\n*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*\n\nWe execute quantitative queries to verify:\n1. **Model Dataset Grain**: 1 row = 1 `content_id` with zero duplicate collisions across 30,000 items.\n2. **Missingness & Availability Traps**: Quantitative measurement of `avg_position == 0` (1,205 rows) and `word_count` missingness (25.7%).\n3. **Warehouse Entity Integration**: Verification of warehouse `dim_clients` (104 clients) and `dim_content` grain integrity.\n

In [3]:
# Section 3 Code: Contract Verification Queries (Grain, Missingness, and Warehouse Checks)
print("=== QUERY 1: THE GRAIN PROBE (content_id uniqueness) ===")
dups = df.groupby('content_id').size().loc[lambda x: x > 1]
print(f" - Duplicate content_id collisions: {len(dups)}")
print(f" - Grain verification status: {'PASSED (1 Row = 1 Content Item)' if len(dups) == 0 else 'FAILED'}")

print("\n=== QUERY 2: SLICE COUNTS & PORTFOLIO SPAN ===")
print(f" - Total Content Rows    : {len(df):,}")
print(f" - Distinct Client Sites : {df['client_id'].nunique()}")
print(f" - Median Impressions 90d: {df['impressions_90d'].median():.0f} (Mean: {df['impressions_90d'].mean():.1f})")
print(f" - Declining Content Items: {(df['is_declining_label'] == 1).sum():,} ({df['is_declining_label'].mean()*100:.1f}%)")

print("\n=== QUERY 3: MISSINGNESS & TRAP AUDIT ===")
zero_pos = (df['avg_position'] == 0).sum()
wc_null = df['word_count'].isnull().sum()
print(f" - Position == 0 (Unranked Pages): {zero_pos:,} rows ({zero_pos/len(df)*100:.2f}%) -> Imputed to 100.0")
print(f" - Word Count Missing            : {wc_null:,} rows ({wc_null/len(df)*100:.2f}%) -> Retained indicator flag")

print("\n=== QUERY 4: WAREHOUSE DIMENSION VERIFICATION (DuckDB) ===")
client_spans = con.sql(f"""
    SELECT 
        COUNT(*) AS clients,
        MIN(gsc_data_start) AS earliest_gsc,
        MAX(gsc_data_start) AS latest_gsc,
        COUNT(*) FILTER (WHERE ga4_data_start IS NOT NULL) AS clients_with_ga4
    FROM {CLIENTS_SRC}
""").df()
print(client_spans.to_string(index=False))


=== QUERY 1: THE GRAIN PROBE (content_id uniqueness) ===
 - Duplicate content_id collisions: 0
 - Grain verification status: PASSED (1 Row = 1 Content Item)

=== QUERY 2: SLICE COUNTS & PORTFOLIO SPAN ===
 - Total Content Rows    : 30,000
 - Distinct Client Sites : 32
 - Median Impressions 90d: 731 (Mean: 5200.4)
 - Declining Content Items: 16,262 (54.2%)

=== QUERY 3: MISSINGNESS & TRAP AUDIT ===
 - Position == 0 (Unranked Pages): 1,205 rows (4.02%) -> Imputed to 100.0
 - Word Count Missing            : 7,699 rows (25.66%) -> Retained indicator flag

=== QUERY 4: WAREHOUSE DIMENSION VERIFICATION (DuckDB) ===


 clients earliest_gsc latest_gsc  clients_with_ga4
     104   2025-01-27 2026-06-02                51


## 4. Data limits\n\n*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*\n\n### Portfolio Boundaries & The Real Leakage Trap Experiment\n\n1. **Portfolio Boundaries**: Multi-client data exhibits heterogeneous histories (`gsc_data_start`), mandating `GroupShuffleSplit` by `client_id` rather than random row-level splitting.\n2. **The Leakage Trap Experiment**:\n   - Train an honest decision-tree/forest model on the 7 pre-decision clustering features predicting `is_declining_label`.\n   - Deliberately inject **ONE** future trend feature (`trend_pct`).\n   - Observe the artificial performance explosion from honest ROC-AUC toward near-perfect 1.00.\n   - Delete the trap column and retain the honest baseline score.\n

In [4]:
# Section 4 Code: Portfolio Boundaries & The Leakage Trap Experiment
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

feature_cols = [
    'log_impressions', 'clean_avg_position', 'ctr', 'days_since_last_update',
    'content_age_days', 'clean_word_count', 'engagement_rate'
]
X_honest = df[feature_cols]
y = df['is_declining_label']
leak_col = df['trend_pct']

X_tr, X_te, y_tr, y_te, leak_tr, leak_te = train_test_split(
    X_honest, y, leak_col, test_size=0.25, random_state=42, stratify=y
)

# 1. Fit Honest Model
clf_honest = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_honest.fit(X_tr, y_tr)
honest_auc = roc_auc_score(y_te, clf_honest.predict_proba(X_te)[:, 1])

# 2. The Trap: Add ONE future trend feature deliberately
X_tr_leaky = X_tr.copy()
X_tr_leaky['LEAK_trend_pct'] = leak_tr
X_te_leaky = X_te.copy()
X_te_leaky['LEAK_trend_pct'] = leak_te

clf_leaky = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf_leaky.fit(X_tr_leaky, y_tr)
leaky_auc = roc_auc_score(y_te, clf_leaky.predict_proba(X_te_leaky)[:, 1])

print("=== LEAKAGE EXPERIMENT RESULTS (Starter Dataset & Evaluative Proxy) ===")
print(f" - Honest Model ROC-AUC : {honest_auc:.4f}")
print(f" - Leaky Model ROC-AUC  : {leaky_auc:.4f} (ARTIFICIAL JUMP TO 1.00 DETECTED!)")
print(f" - Leakage Jump Delta   : +{leaky_auc - honest_auc:.4f}")

del X_tr_leaky['LEAK_trend_pct']
del X_te_leaky['LEAK_trend_pct']
print(f"\n[Action Taken]: Deleted 'trend_pct' from feature set. Retaining honest score: {honest_auc:.4f}")


=== LEAKAGE EXPERIMENT RESULTS (Starter Dataset & Evaluative Proxy) ===
 - Honest Model ROC-AUC : 0.7152
 - Leaky Model ROC-AUC  : 0.9997 (ARTIFICIAL JUMP TO 1.00 DETECTED!)
 - Leakage Jump Delta   : +0.2846

[Action Taken]: Deleted 'trend_pct' from feature set. Retaining honest score: 0.7152


## Self-check\n\nBefore you submit, confirm each line honestly:\n\n- [x] Every section above is filled — markdown thinking AND the code that backs it\n- [x] The notebook runs top to bottom with no errors (Runtime → Run all)\n- [x] No client names, URLs, or private queries anywhere\n- [x] My claims use careful words: observed, measured, directional, decision-support\n- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.\n